# 🤖 Assignment 2: Simple Policy Q&A Bot
**Data Science & GenAI Assessment**

| | |
|---|---|
| **Name** | RAVINDRA KUMAR NAYAK |
| **Roll No.** | 2201MC30 |

---

## 📌 Objective
Build a Q&A system that retrieves relevant information from company policy documents and answers user questions — without using any paid API or GPU-based model.

## 🗂️ Documents Provided

| File | Content |
|------|---------|
| `leave_policy.txt` | Employee leave entitlements (paid, sick, maternity, casual) |
| `it_policy.txt` | IT rules (VPN, passwords, devices, antivirus, USB) |
| `travel_policy.txt` | Travel reimbursement rules (flights, hotels, local travel) |

## 🔁 Workflow
```
Upload Docs → Load & Preprocess Text → Build TF-IDF Vectors
           → User Asks Question → Compute Cosine Similarity
           → Retrieve Best Matching Sentence → Return Answer + Source
```

## 🧠 Approach: TF-IDF + Cosine Similarity

| Term | What it means |
|------|---------------|
| **TF-IDF** | Converts text into numbers — rare important words get higher scores |
| **Cosine Similarity** | Measures how similar two text vectors are (0 = nothing in common, 1 = identical) |
| **Retrieval** | We find the policy sentence most similar to the user's question |

> No paid APIs. No GPU. No large language models. Pure classical NLP — fast, transparent, and fully explainable.

---
## Step 0 — Install & Import Libraries

| Library | Purpose |
|---------|--------|
| `sklearn` | TF-IDF vectorizer and cosine similarity computation |
| `numpy` | Array operations for similarity scores |
| `os` | File path handling |
| `re` | Regular expressions for text cleaning |
| `textwrap` | Wraps long answer text cleanly in the output |

In [1]:
!pip install scikit-learn --quiet

import os
import re
import numpy as np
import textwrap

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print('✅ All libraries imported successfully.')

✅ All libraries imported successfully.


---
## Step 1 — Upload Policy Documents

Just like Assignment 1, we use `google.colab.files.upload()` to bring our local `.txt` files into the Colab session.

**Upload all three files at once** when the file picker appears:
- `leave_policy.txt`
- `it_policy.txt`
- `travel_policy.txt`

> ⚠️ Colab storage resets when the session ends. Re-run this cell and re-upload if you reconnect.

In [2]:
from google.colab import files

print("📁 Select all 3 policy files at once: leave_policy.txt, it_policy.txt, travel_policy.txt")
uploaded = files.upload()

print(f'\n✅ Files uploaded: {list(uploaded.keys())}')

📁 Select all 3 policy files at once: leave_policy.txt, it_policy.txt, travel_policy.txt


Saving it_policy.txt to it_policy.txt
Saving leave_policy.txt to leave_policy.txt
Saving travel_policy.txt to travel_policy.txt

✅ Files uploaded: ['it_policy.txt', 'leave_policy.txt', 'travel_policy.txt']


---
## Step 2 — Load & Preview Policy Documents

We read each `.txt` file into a Python dictionary where:
- **Key** = document name (used later to tell the user *which* policy answered their question)
- **Value** = full raw text content of that file

We then preview each document to confirm it loaded correctly.

In [3]:
# Map clean display names to their filenames
policy_files = {
    'Leave Policy'  : 'leave_policy.txt',
    'IT Policy'     : 'it_policy.txt',
    'Travel Policy' : 'travel_policy.txt'
}

# Load each file into a dictionary: { 'Leave Policy': 'full text...', ... }
documents = {}
for doc_name, filename in policy_files.items():
    with open(filename, 'r', encoding='utf-8') as f:
        documents[doc_name] = f.read()
    print(f'✅ Loaded: {doc_name}  ({len(documents[doc_name])} characters)')

print()
print('─' * 55)

# Preview each document
for doc_name, content in documents.items():
    print(f'\n📄 {doc_name}:')
    print(content)
    print('─' * 55)

✅ Loaded: Leave Policy  (279 characters)
✅ Loaded: IT Policy  (293 characters)
✅ Loaded: Travel Policy  (272 characters)

───────────────────────────────────────────────────────

📄 Leave Policy:
Leave Policy

1. Employees are entitled to 15 paid leaves per year.
2. Sick leave entitlement is 10 days per year.
3. Maternity leave is 26 weeks.
4. Casual leave can be taken for a maximum of 3 consecutive days.
5. Unused paid leaves cannot be carried forward to the next year.

───────────────────────────────────────────────────────

📄 IT Policy:
IT Policy

1. VPN is mandatory for remote work access.
2. Employees must not share their passwords with anyone.
3. Company-issued laptop must be used for official work.
4. All devices must have updated antivirus software installed.
5. External USB devices are not allowed without IT approval.

───────────────────────────────────────────────────────

📄 Travel Policy:
Travel Policy

1. Flight bookings must be economy class.
2. Hotel reimbursement limit i

---
## Step 3 — Split Documents into Sentences

TF-IDF works at the **sentence level**, not the whole-document level. Why?

- If we compare a question to the *entire* document, the relevant information gets diluted by unrelated sentences
- Splitting into sentences allows us to pinpoint the **exact line** that answers the question
- This also makes the answer shorter and more precise

### What we build:
A flat list of all sentences across all three documents, each tagged with its source document name.

```
sentences = [ 'VPN is mandatory for remote work.',  'Sick leave is 10 days...', ... ]
sources   = [ 'IT Policy',                          'Leave Policy',             ... ]
```

The `sources` list keeps track of which document each sentence came from — so we can report it in the answer.

In [4]:
def extract_sentences(text):
    """
    Splits a block of text into individual sentences / meaningful lines.
    Steps:
      1. Split on newlines to get each line
      2. Strip whitespace and bullet-point characters (numbers, dots, dashes)
      3. Keep only non-empty lines that are at least 10 characters long
    """
    lines = text.split('\n')
    sentences = []
    for line in lines:
        # Remove leading numbering like '1.', '2.' and bullet characters
        clean = re.sub(r'^[\d\.\-\*\s]+', '', line).strip()
        if len(clean) > 10:   # Skip headers or very short lines
            sentences.append(clean)
    return sentences


# Build two parallel lists:
#   sentences → the actual text of each sentence
#   sources   → which document that sentence came from
sentences = []
sources   = []

for doc_name, content in documents.items():
    doc_sentences = extract_sentences(content)
    sentences.extend(doc_sentences)
    sources.extend([doc_name] * len(doc_sentences))   # Tag every sentence with its source

print(f'Total sentences extracted: {len(sentences)}')
print()
print('=== All Extracted Sentences ===')
for i, (sent, src) in enumerate(zip(sentences, sources)):
    print(f'  [{i:02d}] [{src}] {sent}')

Total sentences extracted: 17

=== All Extracted Sentences ===
  [00] [Leave Policy] Leave Policy
  [01] [Leave Policy] Employees are entitled to 15 paid leaves per year.
  [02] [Leave Policy] Sick leave entitlement is 10 days per year.
  [03] [Leave Policy] Maternity leave is 26 weeks.
  [04] [Leave Policy] Casual leave can be taken for a maximum of 3 consecutive days.
  [05] [Leave Policy] Unused paid leaves cannot be carried forward to the next year.
  [06] [IT Policy] VPN is mandatory for remote work access.
  [07] [IT Policy] Employees must not share their passwords with anyone.
  [08] [IT Policy] Company-issued laptop must be used for official work.
  [09] [IT Policy] All devices must have updated antivirus software installed.
  [10] [IT Policy] External USB devices are not allowed without IT approval.
  [11] [Travel Policy] Travel Policy
  [12] [Travel Policy] Flight bookings must be economy class.
  [13] [Travel Policy] Hotel reimbursement limit is Rs. 5000 per night.
  [14] [T

---
## Step 4 — Build TF-IDF Vectors

### What is TF-IDF?

**TF-IDF** (Term Frequency – Inverse Document Frequency) converts text into numbers so we can do math on it.

| Component | Formula (simplified) | Meaning |
|-----------|---------------------|---------|
| **TF** | count of word in sentence / total words | How often does this word appear here? |
| **IDF** | log(total sentences / sentences containing word) | How rare is this word across all sentences? |
| **TF-IDF** | TF × IDF | Rare but present words score highest |

**Example:**
- The word `"the"` appears everywhere → low IDF → low TF-IDF score (not useful)
- The word `"maternity"` appears in only 1 sentence → high IDF → high TF-IDF score (very useful)

### Parameters we use:

| Parameter | Value | Why |
|-----------|-------|-----|
| `stop_words='english'` | Removes words like "the", "is", "a" | These carry no meaning for matching |
| `ngram_range=(1,2)` | Considers single words AND two-word phrases | Captures "paid leave", "hotel reimbursement" etc. |
| `min_df=1` | Include all terms even if appearing once | We have few documents so we keep everything |

We also build a **per-document index map** — a dictionary that records which rows in the TF-IDF matrix belong to each policy. This is used in Step 5 to restrict the similarity search to one policy at a time.

In [5]:
# ngram_range=(1,2) captures both single words and two-word phrases
# This helps match 'paid leave' as a unit, not just 'paid' and 'leave' separately
vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    min_df=1
)

# Fit and transform all policy sentences into a single TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(sentences)

# Build per-document index map:
# doc_indices['Leave Policy'] = list of row numbers in tfidf_matrix that belong to Leave Policy
# This lets us search within one policy's sentences only (used in Step 5)
doc_indices = {}
for doc_name in documents.keys():
    doc_indices[doc_name] = [i for i, src in enumerate(sources) if src == doc_name]

print(f'✅ TF-IDF matrix built!')
print(f'   Sentences (rows)       : {tfidf_matrix.shape[0]}')
print(f'   Unique terms (columns) : {tfidf_matrix.shape[1]}')
print()
for doc_name, idxs in doc_indices.items():
    print(f'   {doc_name}: {len(idxs)} sentences  (matrix rows {idxs[0]}–{idxs[-1]})')

✅ TF-IDF matrix built!
   Sentences (rows)       : 17
   Unique terms (columns) : 130

   Leave Policy: 6 sentences  (matrix rows 0–5)
   IT Policy: 5 sentences  (matrix rows 6–10)
   Travel Policy: 6 sentences  (matrix rows 11–16)


---
## Step 5 — Domain Keyword Detection

### Why we need this

The single-stage TF-IDF approach has a critical weakness with a tiny corpus:
common words like `'allowed'` or `'company'` appear across multiple policies
and can mislead the similarity score into picking the wrong document.

**Example of the failure:**
- Question: `'sick leaves allowed'`
- The word `'allowed'` scores high in `'External USB devices are not allowed...'` (IT Policy)
- That sentence beats the correct `'Sick leave entitlement is 10 days'` (Leave Policy)
- Result: wrong answer from the wrong document ❌

### The fix: Two-Stage Retrieval

**Stage 1** — Count how many topic-specific keywords from each policy appear in the question:
```
Question: 'sick leaves allowed'
  Leave Policy score  : 2  ← 'sick' + 'leaves'
  IT Policy score     : 0
  Travel Policy score : 0
  → Route to Leave Policy ✅
```

**Stage 2** — Run TF-IDF similarity **only within that policy's sentences**.
The USB sentence never competes. Correct answer returned every time.

If Stage 1 finds **zero keywords** in any domain, the question is out-of-scope → return fallback immediately without running TF-IDF at all.

In [6]:
# Keywords strongly associated with each policy topic
DOMAIN_KEYWORDS = {
    'Leave Policy': [
        'leave', 'leaves', 'sick', 'maternity', 'casual', 'paid',
        'carry', 'forward', 'entitlement', 'vacation', 'holiday',
        'days', 'weeks', 'annual', 'medical', 'unused', 'consecutive'
    ],
    'IT Policy': [
        'vpn', 'usb', 'laptop', 'password', 'antivirus', 'virus',
        'device', 'remote', 'computer', 'security', 'software',
        'access', 'network', 'official', 'issued'
    ],
    'Travel Policy': [
        'travel', 'hotel', 'flight', 'booking', 'reimbursement',
        'reimburse', 'international', 'local', 'approval', 'expense',
        'economy', 'class', 'night', 'bill', 'bills', 'managerial', 'trip'
    ]
}


def detect_domain(question):
    """
    Stage 1 — Identify which policy domain the question belongs to.
    Counts domain keyword matches in the question.
    Returns (best_domain, score). If score == 0 → out-of-scope.
    """
    q_words = set(re.findall(r'\b\w+\b', question.lower()))

    domain_scores = {
        domain: sum(1 for kw in keywords if kw in q_words)
        for domain, keywords in DOMAIN_KEYWORDS.items()
    }

    best_domain = max(domain_scores, key=domain_scores.get)
    best_score  = domain_scores[best_domain]

    return (best_domain, best_score) if best_score > 0 else (None, 0)


# Quick self-test
test_cases = [
    'How many sick leaves are allowed?',
    'Is VPN required for working from home?',
    'What is the hotel reimbursement limit?',
    'What is the canteen menu today?'
]
print('=== Stage 1 Domain Detection Test ===')
for q in test_cases:
    domain, score = detect_domain(q)
    result = domain if domain else 'None → FALLBACK'
    print(f'  Q : {q}')
    print(f'  →  Domain: {result}  (keyword matches: {score})')
    print()

=== Stage 1 Domain Detection Test ===
  Q : How many sick leaves are allowed?
  →  Domain: Leave Policy  (keyword matches: 2)

  Q : Is VPN required for working from home?
  →  Domain: IT Policy  (keyword matches: 1)

  Q : What is the hotel reimbursement limit?
  →  Domain: Travel Policy  (keyword matches: 2)

  Q : What is the canteen menu today?
  →  Domain: None → FALLBACK  (keyword matches: 0)



---
## Step 6 — Two-Stage Q&A Function

This is the core of the system. Every query follows this logic:

```
Stage 1 — Domain Detection
  Count keyword matches per policy domain in the question
  If zero matches in all domains → return fallback immediately
  Otherwise → select domain with highest keyword count

Stage 2 — Sentence Retrieval (within detected domain only)
  Convert question to TF-IDF vector
  Compare against sentences from the selected domain only
  Find sentence with highest cosine similarity
  If best score < threshold → return fallback
  Otherwise → return answer + source document + confidence score
```

### Why threshold 0.15 (raised from 0.1)?
With Stage 1 already filtering out truly off-topic questions, Stage 2 only runs
on the correct document subset. A slightly higher floor of `0.15` ensures weak
within-domain matches are also rejected cleanly.

### What is Cosine Similarity?
It measures the **angle** between two TF-IDF vectors.
```
Score = 1.0  →  Perfect vocabulary match
Score = 0.5  →  Moderate overlap
Score = 0.0  →  No words in common
```

In [7]:
def answer_question(question, threshold=0.15):
    """
    Two-stage Q&A retrieval.

    Stage 1: Keyword domain detection — routes question to the right policy.
    Stage 2: TF-IDF cosine similarity — finds the best sentence within that policy.

    Parameters:
        question  (str)   : User question in plain English
        threshold (float) : Min cosine similarity to accept an answer (default 0.15)
    """
    print('\n' + '═' * 62)
    print(f'  ❓ Question: {question}')
    print('═' * 62)

    # ── STAGE 1: Domain Detection ──────────────────────────────────
    domain, kw_score = detect_domain(question)

    if domain is None:
        # No policy keywords found — clearly out of scope, skip TF-IDF entirely
        print('  📭 Answer : Information not available in policy documents.')
        print('  📊 Reason : No policy-related keywords detected in the question.')
        print('═' * 62)
        return

    print(f'  🗂️  Domain  : {domain}  (keyword matches: {kw_score})')

    # ── STAGE 2: TF-IDF within the detected domain only ────────────
    question_vec = vectorizer.transform([question])

    # Get only row indices that belong to the detected domain
    domain_rows   = doc_indices[domain]
    # Slice the full matrix to keep only this domain's sentences
    domain_matrix = tfidf_matrix[domain_rows]

    # Cosine similarity: question vector vs every sentence in the domain
    sim_scores = cosine_similarity(question_vec, domain_matrix).flatten()

    best_local_idx  = np.argmax(sim_scores)
    best_score      = sim_scores[best_local_idx]
    best_global_idx = domain_rows[best_local_idx]  # map back to full sentences list

    if best_score < threshold:
        print('  📭 Answer : Information not available in policy documents.')
        print(f'  📊 Best score in {domain}: {best_score:.4f} (below threshold {threshold})')
    else:
        best_answer = sentences[best_global_idx]
        best_source = sources[best_global_idx]
        wrapped     = textwrap.fill(best_answer, width=58)
        print(f'  ✅ Answer : {wrapped}')
        print(f'  📄 Source : {best_source}')
        print(f'  📊 Confidence Score : {best_score:.4f}')

    print('═' * 62)


print('✅ Two-stage Q&A function defined and ready!')

✅ Two-stage Q&A function defined and ready!


---
## Step 7 — Test the Q&A Bot

We test with **9 questions** (3 per policy) plus **2 previously failing cases** that are now fixed.

### What we verify:
- ✅ Correct answers retrieved from the right document
- ✅ Source document correctly identified
- ✅ `'sick leaves allowed'` now returns Leave Policy (not IT Policy)
- ✅ Out-of-scope question now returns fallback (not a wrong match)

### 7.1 — Leave Policy Questions

In [8]:
answer_question('How many paid leaves do employees get per year?')
answer_question('What is the maternity leave duration?')
answer_question('Can I carry forward unused leaves to next year?')


══════════════════════════════════════════════════════════════
  ❓ Question: How many paid leaves do employees get per year?
══════════════════════════════════════════════════════════════
  🗂️  Domain  : Leave Policy  (keyword matches: 2)
  ✅ Answer : Employees are entitled to 15 paid leaves per year.
  📄 Source : Leave Policy
  📊 Confidence Score : 0.6157
══════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════
  ❓ Question: What is the maternity leave duration?
══════════════════════════════════════════════════════════════
  🗂️  Domain  : Leave Policy  (keyword matches: 2)
  ✅ Answer : Maternity leave is 26 weeks.
  📄 Source : Leave Policy
  📊 Confidence Score : 0.6209
══════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════
  ❓ Question: Can I carry forward unused leaves to next year?
════════════════════════════════════════════════════════════

### 7.2 — IT Policy Questions

In [9]:
answer_question('Is VPN required for working from home?')
answer_question('Can I use a USB drive at work?')
answer_question('What device should I use for official work?')


══════════════════════════════════════════════════════════════
  ❓ Question: Is VPN required for working from home?
══════════════════════════════════════════════════════════════
  🗂️  Domain  : IT Policy  (keyword matches: 1)
  ✅ Answer : VPN is mandatory for remote work access.
  📄 Source : IT Policy
  📊 Confidence Score : 0.3378
══════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════
  ❓ Question: Can I use a USB drive at work?
══════════════════════════════════════════════════════════════
  🗂️  Domain  : IT Policy  (keyword matches: 1)
  ✅ Answer : External USB devices are not allowed without IT approval.
  📄 Source : IT Policy
  📊 Confidence Score : 0.2580
══════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════
  ❓ Question: What device should I use for official work?
══════════════════════════════════════════════════════════════
  🗂️  Doma

### 7.3 — Travel Policy Questions

In [10]:
answer_question('What is the hotel reimbursement limit per night?')
answer_question('What class should I book for flights?')
answer_question('Do I need approval for international travel?')


══════════════════════════════════════════════════════════════
  ❓ Question: What is the hotel reimbursement limit per night?
══════════════════════════════════════════════════════════════
  🗂️  Domain  : Travel Policy  (keyword matches: 3)
  ✅ Answer : Hotel reimbursement limit is Rs. 5000 per night.
  📄 Source : Travel Policy
  📊 Confidence Score : 0.7385
══════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════
  ❓ Question: What class should I book for flights?
══════════════════════════════════════════════════════════════
  🗂️  Domain  : Travel Policy  (keyword matches: 1)
  ✅ Answer : Flight bookings must be economy class.
  📄 Source : Travel Policy
  📊 Confidence Score : 0.3780
══════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════
  ❓ Question: Do I need approval for international travel?
══════════════════════════════════════════════════

### 7.4 — Previously Failing Cases (Now Fixed)

These two queries failed in the original single-stage system — both returned
IT Policy's USB sentence because the word `'allowed'` scored higher there.
With Stage 1 routing to Leave Policy first, the correct answer is found.

In [11]:
# These previously returned 'USB devices not allowed' from IT Policy
# Stage 1 now correctly routes both to Leave Policy first
answer_question('Sick leaves allowed')
answer_question('How many sick leaves are allowed')


══════════════════════════════════════════════════════════════
  ❓ Question: Sick leaves allowed
══════════════════════════════════════════════════════════════
  🗂️  Domain  : Leave Policy  (keyword matches: 2)
  ✅ Answer : Sick leave entitlement is 10 days per year.
  📄 Source : Leave Policy
  📊 Confidence Score : 0.1914
══════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════
  ❓ Question: How many sick leaves are allowed
══════════════════════════════════════════════════════════════
  🗂️  Domain  : Leave Policy  (keyword matches: 2)
  ✅ Answer : Sick leave entitlement is 10 days per year.
  📄 Source : Leave Policy
  📊 Confidence Score : 0.1914
══════════════════════════════════════════════════════════════


### 7.5 — Out-of-Scope Fallback Test

These questions have no keywords from any policy domain.  
Stage 1 returns `None` immediately — TF-IDF is never run — and the fallback message is returned.  
The original system returned `'Company-issued laptop...'` with score 0.3048 for the canteen question.

In [12]:
# Original system returned 'Company-issued laptop' (score 0.3048) — now correctly blocked
answer_question('What is the company canteen menu for Monday?')
answer_question('When is the next office party?')


══════════════════════════════════════════════════════════════
  ❓ Question: What is the company canteen menu for Monday?
══════════════════════════════════════════════════════════════
  📭 Answer : Information not available in policy documents.
  📊 Reason : No policy-related keywords detected in the question.
══════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════
  ❓ Question: When is the next office party?
══════════════════════════════════════════════════════════════
  📭 Answer : Information not available in policy documents.
  📊 Reason : No policy-related keywords detected in the question.
══════════════════════════════════════════════════════════════


---
## Step 8 — Transparency View: Top Matches per Domain

This diagnostic tool shows:
1. Which domain Stage 1 selected and the keyword match count
2. The top N sentence candidates **within that domain** and their scores

This makes the two-stage retrieval completely transparent and helps debug edge cases.

In [13]:
def show_top_matches(question, top_n=3):
    """
    Diagnostic view — shows Stage 1 domain selection and
    top N sentence matches within that domain.
    """
    print(f'\n🔍 Query: "{question}"')
    print('─' * 62)

    domain, kw_score = detect_domain(question)

    if domain is None:
        print('  Stage 1 → No domain detected → Fallback (no TF-IDF run)')
        print('─' * 62)
        return

    print(f'  Stage 1 → Domain: {domain}  (keyword matches: {kw_score})')
    print(f'  Stage 2 → Top {top_n} matches within {domain}:')
    print()

    question_vec  = vectorizer.transform([question])
    domain_rows   = doc_indices[domain]
    domain_matrix = tfidf_matrix[domain_rows]
    sim_scores    = cosine_similarity(question_vec, domain_matrix).flatten()

    # argsort ascending, reverse for descending, take top_n
    top_local_idxs = sim_scores.argsort()[::-1][:top_n]

    for rank, local_idx in enumerate(top_local_idxs, 1):
        global_idx = domain_rows[local_idx]
        score      = sim_scores[local_idx]
        text       = textwrap.fill(sentences[global_idx], width=52)
        print(f'  Rank {rank} | Score: {score:.4f}')
        print(f'         {text}')
        print()

    print('─' * 62)


show_top_matches('How many sick leaves are allowed?')
show_top_matches('What are the rules for hotel booking while travelling?')
show_top_matches('What is the canteen menu today?')


🔍 Query: "How many sick leaves are allowed?"
──────────────────────────────────────────────────────────────
  Stage 1 → Domain: Leave Policy  (keyword matches: 2)
  Stage 2 → Top 3 matches within Leave Policy:

  Rank 1 | Score: 0.1914
         Sick leave entitlement is 10 days per year.

  Rank 2 | Score: 0.1476
         Employees are entitled to 15 paid leaves per year.

  Rank 3 | Score: 0.1458
         Unused paid leaves cannot be carried forward to the
next year.

──────────────────────────────────────────────────────────────

🔍 Query: "What are the rules for hotel booking while travelling?"
──────────────────────────────────────────────────────────────
  Stage 1 → Domain: Travel Policy  (keyword matches: 2)
  Stage 2 → Top 3 matches within Travel Policy:

  Rank 1 | Score: 0.3015
         Hotel reimbursement limit is Rs. 5000 per night.

  Rank 2 | Score: 0.0000
         Personal expenses during travel will not be
reimbursed.

  Rank 3 | Score: 0.0000
         International trav

---
## Step 9 — Interactive Q&A Session

Type any question about Leave, IT, or Travel policy and get an answer in real time.
- Press **Enter** to submit
- Type `quit` or `exit` to stop

> This simulates how the bot would work as a real employee-facing tool.

In [14]:
print('🤖 Policy Q&A Bot is ready!')
print('   Ask any question about Leave, IT, or Travel policy.')
print('   Type "quit" or "exit" to stop.\n')

while True:
    user_input = input('Your question: ').strip()

    if not user_input:
        print('  ⚠️  Please enter a question.\n')
        continue

    if user_input.lower() in ['quit', 'exit']:
        print('\n👋 Exiting Q&A Bot. Goodbye!')
        break

    answer_question(user_input)

🤖 Policy Q&A Bot is ready!
   Ask any question about Leave, IT, or Travel policy.
   Type "quit" or "exit" to stop.

Your question: what is in the menu today 

══════════════════════════════════════════════════════════════
  ❓ Question: what is in the menu today
══════════════════════════════════════════════════════════════
  📭 Answer : Information not available in policy documents.
  📊 Reason : No policy-related keywords detected in the question.
══════════════════════════════════════════════════════════════
Your question: how many leaves are allowed 

══════════════════════════════════════════════════════════════
  ❓ Question: how many leaves are allowed
══════════════════════════════════════════════════════════════
  🗂️  Domain  : Leave Policy  (keyword matches: 1)
  ✅ Answer : Employees are entitled to 15 paid leaves per year.
  📄 Source : Leave Policy
  📊 Confidence Score : 0.1848
══════════════════════════════════════════════════════════════
Your question: sick leaves 

═════════

---
## Step 10 — System Summary

A complete confirmation that every assignment requirement has been fulfilled.

In [15]:
total_chars = sum(len(v) for v in documents.values())

print(f"""
╔═══════════════════════════════════════════════════════════════╗
║         ✅ ASSIGNMENT 2 COMPLETE — SYSTEM SUMMARY            ║
╠═══════════════════════════════════════════════════════════════╣

  Documents Loaded
    ✔ Leave Policy    — {len(doc_indices['Leave Policy'])} sentences
    ✔ IT Policy       — {len(doc_indices['IT Policy'])} sentences
    ✔ Travel Policy   — {len(doc_indices['Travel Policy'])} sentences
    ✔ Total           — {len(sentences)} sentences  |  {total_chars} characters

  Retrieval Architecture — Two-Stage
    ✔ Stage 1 : Keyword domain detection (routes to correct policy)
    ✔ Stage 2 : TF-IDF + Cosine Similarity (within domain only)
    ✔ Threshold : 0.15  (rejects weak matches)
    ✔ Source document reported with every answer

  TF-IDF Matrix
    ✔ Rows (sentences)      : {tfidf_matrix.shape[0]}
    ✔ Columns (unique terms) : {tfidf_matrix.shape[1]}

  Test Results
    ✔ Leave policy questions            → answered correctly
    ✔ IT policy questions               → answered correctly
    ✔ Travel policy questions           → answered correctly
    ✔ 'Sick leaves allowed'             → correctly returns Leave Policy answer
    ✔ Out-of-scope canteen question     → fallback message returned correctly

  Constraints Respected
    ✔ No paid APIs used
    ✔ No GPU-based large models used
    ✔ Runs entirely on free Google Colab CPU

╚═══════════════════════════════════════════════════════════════╝
""")


╔═══════════════════════════════════════════════════════════════╗
║         ✅ ASSIGNMENT 2 COMPLETE — SYSTEM SUMMARY            ║
╠═══════════════════════════════════════════════════════════════╣

  Documents Loaded
    ✔ Leave Policy    — 6 sentences
    ✔ IT Policy       — 5 sentences
    ✔ Travel Policy   — 6 sentences
    ✔ Total           — 17 sentences  |  844 characters

  Retrieval Architecture — Two-Stage
    ✔ Stage 1 : Keyword domain detection (routes to correct policy)
    ✔ Stage 2 : TF-IDF + Cosine Similarity (within domain only)
    ✔ Threshold : 0.15  (rejects weak matches)
    ✔ Source document reported with every answer

  TF-IDF Matrix
    ✔ Rows (sentences)      : 17
    ✔ Columns (unique terms) : 130

  Test Results
    ✔ Leave policy questions            → answered correctly
    ✔ IT policy questions               → answered correctly
    ✔ Travel policy questions           → answered correctly
    ✔ 'Sick leaves allowed'             → correctly returns Leave Pol